# 2. Readers, binners, and spatial coordinates

An image context combines three independent components: a reader for sparse spectra, a forward binner for a regular model input grid, and an optional inverse binner for converting decoded grids back to sparse `(m/z, intensity)` pairs.

In [ ]:
import os 
from pathlib import Path

# seting global dir
cwd=Path.cwd()
if cwd.name == "tutorials":
    # os.chdir(cwd.parent.parent) 
    os.chdir(cwd.parent.parent.parent) 
os.getcwd()

In [ ]:
from msi_autoencoder_wrapper.core.wrapper import MSIAutoEncoderWrapper

wrapper = MSIAutoEncoderWrapper(
    project_path="data/tutorial_workspace",
    coordinate_order="xy",
)
image_path = Path("data/tutorial_workspace/imgs/example.imzML").resolve()
assert image_path.is_file(), "Download/copy the example imzML and ibd pair first."

In [ ]:
wrapper.workspace.set_default_image_path('example')

## Discover and configure components

The discovery methods print registered names and constructor parameters in one consistent format. Registered names plus JSON-compatible parameters are preferred: they can be serialized and reconstructed when a model is loaded.

In [ ]:
wrapper.context_manager.get_available_inverse_binners()

In [ ]:
wrapper.context_manager.get_available_readers()
wrapper.context_manager.get_available_binners()
wrapper.context_manager.get_available_inverse_binners()

# The default image was configured above, so the path can be omitted.
reader = wrapper.context_manager.set_reader("PyImzMLReader")
# Without a default image, use the explicit equivalent instead:
# reader = wrapper.context_manager.set_reader("PyImzMLReader", str(image_path))
binner = wrapper.context_manager.set_binner(
    "LinearBinning",
    # bin_step is the requested model-grid resolution. To reuse the
    # reader's native spacing, inspect np.median(np.diff(reader.GetXAxis())).
    bin_step=0.1,
)
inverse = wrapper.context_manager.set_inverse_binner(
    "TopPeaksInverseBinner",
    max_bins=1500,
    window_size=3,
)

`PyImzMLReader` and `M2aiaReader` implement the same wrapper reader contract, so either can back datasets and training. M²aia uses native code and is the preferred performance-oriented backend where its binaries are supported. The pure-Python pyimzML backend is the portability fallback, including macOS installations where M²aia is unavailable. Treat performance as dataset- and installation-dependent and benchmark the intended workflow.

Ready instances and classes are also accepted. This is useful during development, but the constructor still has to expose a complete serializable config if the context is later saved.

In [ ]:
from msi_autoencoder_wrapper.readers.strategies.pyimzml_reader import PyImzMLReader
from msi_autoencoder_wrapper.binners.binners_strategies.linear_binner import LinearBinning

# Ready instances are an alternative to registered names. The same cell can
# be rerun: components in the existing image bucket are replaced in place.
wrapper.workspace.set_active_image(str(image_path))
custom_reader = PyImzMLReader(image_path, active_context=wrapper.active_context)
wrapper.context_manager.set_reader(custom_reader, str(image_path))
custom_binner = LinearBinning(bin_step=0.1, active_context=wrapper.active_context)
wrapper.context_manager.set_binner(custom_binner, str(image_path))

## Raw, binned, and inverse-binned spectra

Readers return the original sparse axis and intensities. A binner sums signal into a fixed grid required by a neural network. An inverse binner is lossy: it selects a bounded set of important grid bins; it cannot restore information discarded during binning.

In [ ]:
wrapper.active_context.reader[0:2, 5:8]

In [ ]:
xs, ys = wrapper.active_context.reader[0]
xs, ys 

In [ ]:
grid_ys = wrapper.active_context.binner(xs=xs, ys=ys)
grid_ys

In [ ]:
restored_xs, restored_ys = wrapper.active_context.inverse_binner(grid_ys)
restored_xs, restored_ys

## Indexing, coordinates, and slices

An integer selects a flat spectrum index. A coordinate tuple selects one pixel. A normal Python slice selects flat indices, while a tuple containing slices selects coordinate values. Slice bounds are coordinate values, not zero-based matrix offsets.

In [ ]:
reader = wrapper.active_context.reader
first_spectrum = reader[0]
first_ten = reader[:10]
x, y, z = reader.GetSpectrumPosition(0)
same_spectrum = reader[(x, y, z)]
region = reader[(slice(x, x + 5), slice(y, y + 5), z)]
print(len(first_ten), len(region))

`coordinate_order="xy"` exposes `(x, y, z)`. Matrix-oriented code often reads more naturally as `(row, column, z)`, which reverses the first two stored axes. The setting is wrapper-wide and affects original and latent readers consistently; it can be changed at runtime.

In [ ]:
wrapper.set_coordinate_order("matrix")
row, column = y, x
matrix_spectrum = reader[(row, column, z)]
matrix_region = wrapper.active_context.get_region(
    slice(row, row + 5),
    slice(column, column + 5),
    z,
    source="image",
)
matrix_region

## Presets and reproducible customization

Reader and binner setup is stored as component names plus parameters. Architecture presets use the same principle and are covered in Tutorial 3. Custom registered components and presets should retain every constructor parameter in their config; see [Custom models](../../../docs/CUSTOM_MODELS.md).

> **Important considerations — TODO:** add domain-specific guidance for profile/centroid data, normalization choices, calibration, and future reader methods.